# 1 — Imports & Environment Check

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models

from tqdm.auto import tqdm

# Confirm GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch version: {torch.__version__}")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
PyTorch version: 2.10.0+cu128


# 2 — Configuration

In [7]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR  = Path("/kaggle/working")          # where train/val/test CSVs live
IMAGE_DIR = Path("/kaggle/input/datasets/miguelmirandar/deepfashion-multimodal/images/images")
CKPT_DIR  = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Task definition ─────────────────────────────────────────────────────────
TASKS = {
    "sleeve_length": 5,   # classes 0-4  (NA=5 excluded)
    "upper_fabric":  7,   # classes 0-6  (NA=7 excluded)
    "upper_color":   7,   # classes 0-6  (NA=7 excluded)
}

# ── Hyperparameters ─────────────────────────────────────────────────────────
CFG = {
    "img_size":      224,
    "batch_size":    64,
    "num_workers":   2,
    "epochs":        20,
    "lr":            3e-4,
    "weight_decay":  1e-2,
    "label_smoothing": 0.1,
    "seed":          42,
}

print("Tasks:", TASKS)
print("Config:", CFG)
print(f"Checkpoint dir: {CKPT_DIR}")

Tasks: {'sleeve_length': 5, 'upper_fabric': 7, 'upper_color': 7}
Config: {'img_size': 224, 'batch_size': 64, 'num_workers': 2, 'epochs': 20, 'lr': 0.0003, 'weight_decay': 0.01, 'label_smoothing': 0.1, 'seed': 42}
Checkpoint dir: /kaggle/working/checkpoints


# 3 — Reproducibility & Class Weights

In [8]:
import random

def seed_everything(seed: int):
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["seed"])

# ── Load class weights saved in Notebook 02 ─────────────────────────────────
weights = torch.load(DATA_DIR / "class_weights.pt", map_location=device)

# Expected keys: "sleeve_length", "upper_fabric", "upper_color"
print("Class weights loaded:")
for task, w in weights.items():
    print(f"  {task}: {w.tolist()}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/class_weights.pt'